# 📊 Evaluasi RAG Chatbot SINEMA
## Universitas Tadulako

Notebook ini mengevaluasi performa sistem RAG chatbot untuk SINEMA.

### Metrik Evaluasi:
1. **Retrieval Accuracy** - Apakah dokumen relevan ter-retrieve
2. **Response Latency** - Waktu respons keseluruhan
3. **Answer Quality** - Kualitas jawaban (manual check)
4. **Context Utilization** - Seberapa baik LLM menggunakan konteks

---
## 1️⃣ Setup

In [5]:
!pip install tabulate
import sys
sys.path.insert(0, r'D:\Alisha\sinema-chatbot')

import time
import pandas as pd
from pathlib import Path
from tabulate import tabulate
from tqdm.auto import tqdm

# Import RAG components
from app.config import DOCUMENTS_DIR, VECTOR_DB_DIR, MODEL_DIR, EMBEDDING_MODEL, SYSTEM_PROMPT
from app.rag import RAGRetriever
from app.llm import LLMGenerator

print('✅ Imports successful!')

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


ModuleNotFoundError: No module named 'app'

In [ ]:
# Initialize RAG Retriever
print('Initializing RAG Retriever...')
retriever = RAGRetriever(
    documents_dir=DOCUMENTS_DIR,
    vector_db_dir=VECTOR_DB_DIR,
    embedding_model_name=EMBEDDING_MODEL
)
retriever.initialize(force_reload=True)  # Rebuild index
print(f'Index ready with {len(retriever.vector_store.documents)} chunks')

In [ ]:
# Initialize LLM Generator
print('Initializing LLM Generator...')
generator = LLMGenerator(model_path=MODEL_DIR)
# This will lazy load the model on first generate call
print('LLM ready!')

---
## 2️⃣ Test Queries

In [ ]:
TEST_QUERIES = [
    {"id": "Q1", "query": "Bagaimana cara mengisi KRS online?", "expected_source": "panduan_krs.md"},
    {"id": "Q2", "query": "Apa saja persyaratan sidang skripsi?", "expected_source": "prosedur_skripsi.md"},
    {"id": "Q3", "query": "Jenis beasiswa apa yang tersedia?", "expected_source": "informasi_beasiswa.md"},
    {"id": "Q4", "query": "Bagaimana cara membayar UKT secara online?", "expected_source": "panduan_ukt.md"},
    {"id": "Q5", "query": "Dimana lokasi kantor administrasi akademik?", "expected_source": "layanan_akademik.md"},
    {"id": "Q6", "query": "Berapa batas maksimal SKS yang bisa diambil?", "expected_source": "panduan_krs.md"},
    {"id": "Q7", "query": "Bagaimana cara mengajukan cuti kuliah?", "expected_source": "layanan_akademik.md"},
    {"id": "Q8", "query": "Apa itu beasiswa KIP Kuliah?", "expected_source": "informasi_beasiswa.md"},
]

print(f'Test queries prepared: {len(TEST_QUERIES)} questions')
print(tabulate(pd.DataFrame(TEST_QUERIES)[['id', 'query']], headers='keys', tablefmt='grid', showindex=False))

---
## 3️⃣ Evaluate Retrieval

In [ ]:
retrieval_results = []

print('Evaluating retrieval...')
for q in tqdm(TEST_QUERIES):
    results = retriever.retrieve(q['query'], top_k=3)
    
    # Check if expected source is in top results
    sources = [doc.metadata.get('source', '') for doc, score in results]
    scores = [score for doc, score in results]
    
    hit = q['expected_source'] in sources
    top_source = sources[0] if sources else 'N/A'
    top_score = scores[0] if scores else 0
    
    retrieval_results.append({
        'id': q['id'],
        'query': q['query'][:40] + '...',
        'expected': q['expected_source'],
        'top_source': top_source,
        'top_score': round(top_score, 3),
        'hit': '✅' if hit else '❌'
    })

df_retrieval = pd.DataFrame(retrieval_results)
print('\n📊 RETRIEVAL RESULTS')
print(tabulate(df_retrieval, headers='keys', tablefmt='grid', showindex=False))

accuracy = sum(1 for r in retrieval_results if r['hit'] == '✅') / len(retrieval_results) * 100
print(f'\n🎯 Retrieval Accuracy: {accuracy:.1f}%')

---
## 4️⃣ Evaluate End-to-End RAG

In [ ]:
e2e_results = []

print('Evaluating end-to-end RAG...')
for q in tqdm(TEST_QUERIES):
    # Get context
    context = retriever.get_context(q['query'])
    
    # Generate response
    start = time.time()
    response, latency = generator.generate(
        query=q['query'],
        context=context,
        system_prompt=SYSTEM_PROMPT
    )
    
    e2e_results.append({
        'id': q['id'],
        'query': q['query'][:30] + '...',
        'latency': round(latency, 2),
        'response_len': len(response),
        'has_context': '✅' if context else '❌',
        'response_preview': response[:100] + '...'
    })

df_e2e = pd.DataFrame(e2e_results)
print('\n📊 END-TO-END RESULTS')
print(tabulate(df_e2e[['id', 'query', 'latency', 'response_len', 'has_context']], headers='keys', tablefmt='grid', showindex=False))

avg_latency = df_e2e['latency'].mean()
print(f'\n⚡ Average Latency: {avg_latency:.2f}s')

---
## 5️⃣ Sample Responses

In [ ]:
print('📝 SAMPLE RESPONSES')
print('='*80)
for i, r in enumerate(e2e_results[:3]):
    print(f'\n[{r["id"]}] {TEST_QUERIES[i]["query"]}')
    print('-' * 40)
    print(r['response_preview'])
    print()

---
## 6️⃣ Summary

In [ ]:
print('='*60)
print('📊 EVALUATION SUMMARY')
print('='*60)
print(f'Total Test Queries: {len(TEST_QUERIES)}')
print(f'Retrieval Accuracy: {accuracy:.1f}%')
print(f'Average Latency: {avg_latency:.2f}s')
print(f'Documents Indexed: {len(retriever.vector_store.documents)} chunks')
print('='*60)

# Recommendations
print('\n💡 RECOMMENDATIONS:')
if accuracy < 80:
    print('- Consider adding more documents or improving chunking')
if avg_latency > 5:
    print('- Consider using a smaller model or GPU optimization')
if accuracy >= 80 and avg_latency <= 5:
    print('- System is performing well! Ready for production.')